# Собственная CNN-классификатор символов для OCR

**Цель (для защиты):** показать, что умею строить свою нейросеть, не только тюнить чужие.

**Связь с ДЗ:**
- **ДЗ13** — baseline → improved паттерн (сначала простая сеть, потом усложнённая)
- **ДЗ14** — BatchNorm + Dropout + MaxPooling для борьбы с overfitting
- **ДЗ15** — оценка по per-class метрикам (confusion matrix)

**Датасет:** Synthetic Ukrainian LP от Zenodo (10 000 изображений номеров с character-level YOLO-аннотациями). Извлекаем кропы отдельных символов → классификация на 24 класса.

In [ ]:
import sys, yaml
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import cv2, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from tqdm.auto import tqdm

from src.char_cnn import CharCNN, CHAR_CLASSES, NUM_CLASSES, class_idx

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| classes:', NUM_CLASSES)

CFG = yaml.safe_load(open('../config.yaml', encoding='utf-8'))
SYN = Path(CFG['data_root']) / 'synthetic_ua'

## 1. Извлечение символов из Zenodo-датасета

Каждое фото — номер 193×72, txt содержит YOLO-бокс каждого символа.
Извлекаем кропы символов, ресайзим до 32×32, сохраняем в память.

In [ ]:
def extract_chars(split: str):
    img_dir = SYN / split / 'images'
    lbl_dir = SYN / split / 'labels'
    X, y = [], []
    for img_path in tqdm(sorted(img_dir.glob('*.png')), desc=split):
        # Имя файла = сам номер (AA0000BC.png → AA0000BC)
        plate = img_path.stem.upper()
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        H, W = img.shape
        lbl = (lbl_dir / f'{img_path.stem}.txt').read_text().strip().splitlines()
        # Каждая строка: class_id cx cy bw bh (YOLO norm)
        # Сортируем по cx (слева направо) чтобы сопоставить с именем файла.
        boxes = [list(map(float, l.split())) for l in lbl]
        boxes.sort(key=lambda b: b[1])
        if len(boxes) != len(plate): continue
        for ch, (cid, cx, cy, bw, bh) in zip(plate, boxes):
            x1 = max(0, int((cx - bw/2) * W))
            y1 = max(0, int((cy - bh/2) * H))
            x2 = min(W, int((cx + bw/2) * W))
            y2 = min(H, int((cy + bh/2) * H))
            crop = img[y1:y2, x1:x2]
            if crop.size == 0 or ch not in CHAR_CLASSES: continue
            crop = cv2.resize(crop, (32, 32), interpolation=cv2.INTER_AREA)
            X.append(crop); y.append(class_idx(ch))
    return np.array(X), np.array(y)

X_tr, y_tr = extract_chars('train')
X_va, y_va = extract_chars('valid')
print(f'train: {X_tr.shape}, {np.bincount(y_tr, minlength=NUM_CLASSES).min()}..{np.bincount(y_tr, minlength=NUM_CLASSES).max()} per class')
print(f'valid: {X_va.shape}')

## 2. Балансировка и визуальная проверка (ДЗ12 паттерн)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3))
ax[0].bar(range(NUM_CLASSES), np.bincount(y_tr, minlength=NUM_CLASSES))
ax[0].set_xticks(range(NUM_CLASSES)); ax[0].set_xticklabels(CHAR_CLASSES)
ax[0].set_title('Распределение классов (train)')
# Примеры кропов
samples_per_class = 1
grid = []
for c in range(NUM_CLASSES):
    idx = np.where(y_tr == c)[0]
    if len(idx):
        grid.append(X_tr[idx[0]])
ax[1].imshow(np.hstack(grid), cmap='gray'); ax[1].set_title('По 1 примеру каждого класса'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 3. Dataset / DataLoader с аугментацией (ДЗ14 стиль)

In [ ]:
class CharDataset(Dataset):
    def __init__(self, X, y, train=False):
        self.X = X; self.y = y; self.train = train
        self.tf = T.Compose([
            T.ToPILImage(),
            T.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)) if train else T.Lambda(lambda x: x),
            T.ColorJitter(brightness=0.2, contrast=0.2) if train else T.Lambda(lambda x: x),
            T.ToTensor(),   # → (1, 32, 32) / 255
        ])
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return self.tf(self.X[i]), int(self.y[i])

train_dl = DataLoader(CharDataset(X_tr, y_tr, train=True), batch_size=128, shuffle=True, num_workers=0)
val_dl   = DataLoader(CharDataset(X_va, y_va, train=False), batch_size=256, shuffle=False, num_workers=0)

## 4. Обучение

In [ ]:
model = CharCNN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)
crit = nn.CrossEntropyLoss()

EPOCHS = 15
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    # train
    model.train(); total = 0; n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        out = model(xb); loss = crit(out, yb)
        loss.backward(); opt.step()
        total += loss.item() * xb.size(0); n += xb.size(0)
    tr_loss = total / n

    # val
    model.eval(); total = 0; correct = 0; n = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb); loss = crit(out, yb)
            total += loss.item() * xb.size(0); n += xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
    val_loss = total / n; val_acc = correct / n
    sched.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'epoch {epoch:02d}  train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(history['val_acc']); ax[1].set_title('val accuracy'); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 5. Per-class метрики (ДЗ15 стиль) — confusion matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
preds, truths = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        out = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        preds.extend(out); truths.extend(yb.numpy())

print(classification_report(truths, preds, target_names=CHAR_CLASSES, digits=3))

cm = confusion_matrix(truths, preds, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CHAR_CLASSES, yticklabels=CHAR_CLASSES, ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion matrix (val)')
plt.tight_layout(); plt.show()

## 6. Сохранить веса

In [ ]:
out = Path('../models/char_cnn.pt')
torch.save({'state_dict': model.state_dict(), 'classes': CHAR_CLASSES}, out)
print('Saved to', out, f'({out.stat().st_size/1e6:.2f} MB)')

## Выводы

**Архитектурные решения, пришедшие из ДЗ:**
- Из ДЗ14 взят паттерн «два подряд Conv→BN→ReLU, затем MaxPool и Dropout» — это классический способ стабилизировать обучение и бороться с overfitting.
- Из ДЗ13/15 — оценка через per-class метрики: где сеть путается (например, 0↔O, I↔1).
- Аугментация (поворот, сдвиг, jitter) — страховка против переобучения на синтетике (см. ДЗ14).

**Роль в дипломе:** готовый модуль, который можно подключить в `ALPRPipeline.recognize()` как альтернативу EasyOCR. В отчёте сравним CER/accuracy двух подходов.